# Study 920 — Total Cost of Ownership 🧾

**Cheapest fee or tightest spread — at what holding period does each win?**

Two funds hold the same index. One is the mega-liquid original with the higher fee and the
tightest quote (SPY at 9.45 bp, QQQ at 20.0 bp). The other is the cheap clone
with a slightly wider quote (IVV and VOO at 3.0 bp, QQQM at 15.0 bp). The
cheap one wins a little every day you hold it and loses a little the day you buy it. So there
is a **break-even holding period**, and this study finds it.

Total cost = **expense ratio** (a prospectus number) + **realised tracking difference** (the
only piece actually on the tape) + **round-trip spread** (a quote-level number). We measure the
middle term on daily **total-return** closes and sweep the other two.

*Real-tape numbers below are the frozen headline (`docs/results.md`, common window
2020-10-13 → 2026-06-30, 1,434 days, fingerprint `c3aa747fad51`,
as-of 2026-06-30). The live cells run only the offline synthetic control and the break-even
arithmetic, and say so.*


## 1. The sticker price is not the price

Every fund publishes an expense ratio, and almost nobody checks whether that is what the fund actually cost them. It usually is not. A wrapper can lend its shares out and keep or hand back the revenue; it can hold a sliver of uninvested cash; it can be a 1993-vintage *unit investment trust* that is legally forbidden from reinvesting dividends until the next quarterly payout. All of that lands in one measurable number: the **realised tracking difference** — how much more (or less) one fund actually delivered than its twin on the same index.

> 🔬 **For the quants.** We measure it as the drift of `log(cheap / liquid)` on total-return closes, chained across complete calendar years. Total return is not optional: on price-only closes two funds with different dividend schedules would look like they had different tracking differences when they merely pay on different days.

## 2. What the tape says

Over the 5 complete years in which all these funds trade side by side (2020-10-13 → 2026-06-30):

| The cheap fund, over its pricier twin | Its fee advantage says | The tape delivered | *t* |
|---|--:|--:|--:|
| IVV over SPY | +6.45 bp/yr | **+5.78 bp/yr** | +2.92 |
| VOO over SPY | +6.45 bp/yr | **+6.60 bp/yr** | +3.40 |
| QQQM over QQQ | +5.00 bp/yr | **+7.19 bp/yr** | +5.27 |
| *VOO over IVV — both charge 3 bp* | +0.00 bp/yr | *+0.82 bp/yr* | *+0.75* |

The advantage is real and it is roughly the size the prospectus implies. Each of the three cheap funds beat its twin in **all five years** — QQQM's were +2.2, +8.8, +10.1, +8.4, +6.5 bp — while the same-fee pair managed only three of five.

> ⚠️ **One thing this table cannot tell you: *why*.** SPY (1993) and QQQ (1999) are *unit investment trusts*, a legal form that cannot reinvest dividends between quarterly payouts. So they lose ground for two reasons at once — the higher fee *and* the idle cash — and no pair on the tape separates them. What you are looking at is what the cheap wrapper **delivered**, not what the fee **cost**.

## 3. The last row is the important one

VOO and IVV charge the **same** 3 basis points. If our ruler were broken — if it manufactured a tracking difference out of noise — that pair would show one. It does not: **+0.82 bp/yr**, *t* = +0.75, and an interval that comfortably includes zero.

That is what a placebo is for. The ruler fires where a fee gap exists and stays quiet where none does, on the same tape, in the same window, with the same code.

## 4. So: how long must you hold?

The cheap fund saves you about **7 basis points a year**, forever. Buying it costs you the extra spread, **once**. Divide one by the other and you get the break-even. The cell below does exactly that arithmetic — it is a division, not a backtest, so it runs live.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tco import strategy as st

# Realised tracking differences, frozen from docs/results.md (real tape).
td = {'IVV over SPY': 5.7816, 'VOO over SPY': 6.6036, 'QQQM over QQQ': 7.1898,
      'VOO over IVV (placebo)': 0.822}
print('extra round-trip spread you pay for the cheap fund ->  break-even holding period')
for name, t in td.items():
    row = '  '.join(
        ('never' if st.breakeven_days(t, s) == float('inf')
         else '%5.0f d' % st.breakeven_days(t, s)) for s in (0.5, 1.0, 2.0, 5.0))
    print('%-24s (%+.2f bp/yr):  %s' % (name, t, row))
print('\ncolumns: 0.5 bp   1 bp    2 bp    5 bp  of extra round-trip spread')

extra round-trip spread you pay for the cheap fund ->  break-even holding period
IVV over SPY             (+5.78 bp/yr):     22 d     44 d     87 d    218 d
VOO over SPY             (+6.60 bp/yr):     19 d     38 d     76 d    191 d
QQQM over QQQ            (+7.19 bp/yr):     18 d     35 d     70 d    175 d
VOO over IVV (placebo)   (+0.82 bp/yr):    153 d    307 d    613 d   1533 d

columns: 0.5 bp   1 bp    2 bp    5 bp  of extra round-trip spread


**About two months.** At a one basis-point wider round trip — a generous assumption for funds this liquid — the cheap wrapper has repaid itself in 35–44 trading days. Even at a punitive five basis points it repays inside nine months. And the placebo pair, correctly, takes 307 days to repay a spread it has no advantage to repay it with.

> ⚠️ **Those are point estimates, and the tape is not precise.** Five years of data pin a six basis-point number down only loosely. At the pessimistic end of an honest interval the same one basis-point break-even stretches to **74 days for QQQM/QQQ, 208 for VOO/SPY and 898 for IVV/SPY** — three and a half years for that last pair — and to *never* for the placebo. The direction is well established; the *size* is not.

## 5. Does the tape agree with the arithmetic?

Dividing one number by another is easy. So we also walked every overlapping window on the real tape — buy at tomorrow's close, hold *H* days, pay one extra basis point of round trip — and asked when the cheap fund actually started winning. QQQM over QQQ:

| Hold | Average edge | Share of windows won | *t* |
|---|--:|--:|--:|
| 21 d | -0.51 bp | 44% | -3.05 |
| 63 d | +0.73 bp | 61% | +2.68 |
| 126 d | +2.65 bp | 79% | +5.19 |
| 252 d | +7.47 bp | 92% | +6.11 |
| 504 d | +19.61 bp | 99% | +6.65 |
| 756 d | +36.81 bp | 99% | +10.43 |
| 1008 d | +48.91 bp | 99% | +57.30 |

Positive from **42 days**, statistically clear from **63 days**, and by two years the cheap wrapper wins 99% of all windows. The arithmetic and the tape agree.

The placebo pair, over the same horizons, does **not** repay that one basis point: it is still -0.71 bp at two years and -0.67 bp at three, turning marginally positive (+0.18 bp) only at four.

## 6. Where this stops being worth anything

Two honest limits.

**It is not a trade.** If you try to *harvest* the gap — long the cheap fund, short the expensive one — you capture only +1.93 bp/yr against 90 bp/yr of day-to-day tracking noise, and the borrow you pay on the short leg buries it: -3.77 bp/yr at a five basis-point borrow, -23.77 at twenty-five. This is a saving you *own*, not one you trade.

**It is small.** Six basis points a year is £6 per £10,000. Real, certain, and never worth churning a position to capture. If you are already holding the expensive wrapper in a taxable account with a large gain, the tax bill on switching dwarfs decades of the saving.

## 7. The ruler, checked against a world we built

The cell below is **synthetic** — a made-up pair of funds with a fee gap we planted ourselves. It is here only to show the measuring instrument is honest: it must recover a gap that is there, and report nothing when there is none.

In [2]:
from tco import data
for planted in (0.0, 6.0):
    px, _ = data.synthetic_daily(gap_bp_yr=planted,
                                 signal_strength=1.0 if planted else 0.0, seed=920)
    d = st.synthetic_detect(px, n_boot=400)
    print('planted %5.1f bp/yr -> recovered %+6.2f bp/yr  (t %+5.2f, 95%% CI [%+.2f, %+.2f])'
          % (planted, d['td_ann_bp_yr'], d['t_annual'], d['ci_low'], d['ci_high']))

planted   0.0 bp/yr -> recovered  -0.04 bp/yr  (t -0.02, 95% CI [-3.15, +2.97])


planted   6.0 bp/yr -> recovered  +6.17 bp/yr  (t +2.82, 95% CI [+3.06, +9.18])


## Verdict

- **Signal — Real.** The cheap wrapper's advantage is on the tape at **+5.78 / +6.60 / +7.19 bp/yr** (*t* = +2.92 / +3.40 / +5.27), positive in **5/5 years on each pair**, the size the prospectuses imply, and absent from the same-fee placebo pair.
- **Tradability — Investable.** The break-even is about **two months** at a one basis-point spread differential and under nine months at five. No timing, no leverage, no forecast — just own the cheaper wrapper, which is why it is bankable even though the tape measures it loosely. The prize is a basis point a month, so never churn to get it, and never try to short the gap.
- **Three caveats we will not bury.** *(1)* The expensive fund in every winning pair is a **unit investment trust**, so part of the gap is idle dividend cash rather than the fee, and nothing here separates the two. *(2)* Measured over the *full* histories rather than the common window, the S&P pairs miss significance (+0.78 and +0.69) — because the older public price series contains adjustment errors worth **-56 bp in a single year** between two funds that charge *identical* fees. *(3)* Even on the common window, whether you see the gap at all depends on measuring it year-end to year-end: the raw start-to-finish drift puts the same-fee placebo pair (+4.69 bp/yr) **above** the genuine IVV/SPY pair (+3.38). The tape is dirtier than the thing being measured; the quant notebook performs the autopsy.